# Finland Integrated Train-Weather Dataset

Overview
Comprehensive dataset integrating Finnish railway operational data with meteorological observations for delay prediction research. Addresses the gap in publicly available datasets combining weather information with Finland railway operations.

Each record represents a single timetable event (arrival or departure) of a long-distance train at a station, enriched with weather observations from the geographically closest FMI weather station (EMS) at the time of the stop.

## Project Scope: 

This project uses the 2025 subset of the Finland Integrated Train-Weather Dataset (FI-TW), 
selected from the larger multi-year dataset to provide a manageable and consistent one-year dataset for train delay prediction.

In [1]:
# Step 0 - Import libraries
# --------------------------

# Import the libraries needed for this section
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from imblearn.under_sampling import RandomUnderSampler

SEED=1

In [2]:
# Location of the raw data inside the repository
data_dir = Path("../data/raw")

# Find all 2025 monthly Parquet files
files_2025 = sorted(
    data_dir.glob("matched_data_flat_2025_*.parquet")
)

print(f"Found {len(files_2025)} files")

# Load each monthly file
dfs = [pd.read_parquet(f) for f in files_2025]

print(f"Successfully loaded {len(dfs)} files")

# Combine all 12 monthly files into one 2025 dataset for consistent analysis
data_2025 = pd.concat(
    dfs,
    ignore_index=True
)

print(f"Combined dataset shape: {data_2025.shape}")

Found 12 files
Successfully loaded 12 files
Combined dataset shape: (6284074, 125)


In [3]:
del dfs

In [4]:
data_2025.head()

,trainNumber,departureDate,operatorUICCode,operatorShortCode,trainType,trainCategory,commuterLineID,runningCurrently,cancelled,version,...,Precipitation amount (12h cumulative),Precipitation amount (24h mean),Precipitation amount (24h cumulative),Precipitation amount (72h mean),Precipitation amount (72h cumulative),estimateSource,liveEstimateTime,stopSector,unknownTrack,unknownDelay
0,1,2025-01-01,10,vr,IC,Long-distance,NaN,False,False,290136848769,...,1.6,0.03,1.6,0.12,17.3,None,None,None,NaN,NaN
1,1,2025-01-01,10,vr,IC,Long-distance,NaN,False,False,290136848769,...,2.7,0.06,2.7,0.17,24.2,None,None,None,NaN,NaN
2,1,2025-01-01,10,vr,IC,Long-distance,NaN,False,False,290136848769,...,2.7,0.06,2.7,0.17,24.2,None,None,None,NaN,NaN
3,1,2025-01-01,10,vr,IC,Long-distance,NaN,False,False,290136848769,...,3.8,0.08,3.8,0.18,25.3,None,None,None,NaN,NaN
4,1,2025-01-01,10,vr,IC,Long-distance,NaN,False,False,290136848769,...,3.8,0.08,3.8,0.18,25.3,None,None,None,NaN,NaN


In [5]:
data_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6284074 entries, 0 to 6284073
Columns: 125 entries, trainNumber to unknownDelay
dtypes: bool(4), float64(96), int64(4), object(21)
memory usage: 5.7+ GB


In [6]:
pd.set_option("display.max_columns", None)

data_2025.columns.tolist()


['trainNumber',
 'departureDate',
 'operatorUICCode',
 'operatorShortCode',
 'trainType',
 'trainCategory',
 'commuterLineID',
 'runningCurrently',
 'cancelled',
 'version',
 'timetableType',
 'timetableAcceptanceDate',
 'stationName',
 'stationShortCode',
 'stationUICCode',
 'countryCode',
 'type',
 'trainStopping',
 'commercialStop',
 'stop_cancelled',
 'scheduledTime',
 'actualTime',
 'differenceInMinutes',
 'differenceInMinutes_offset',
 'differenceInMinutes_eachStation_offset',
 'causes',
 'commercialTrack',
 'trainReady',
 'closest_ems',
 'Air temperature',
 'Wind speed',
 'Gust speed',
 'Wind direction',
 'Relative humidity',
 'Dew-point temperature',
 'Precipitation amount',
 'Precipitation intensity',
 'Snow depth',
 'Pressure (msl)',
 'Horizontal visibility',
 'Cloud amount',
 'Present weather (auto)',
 'Air temperature (12h max)',
 'Air temperature (12h min)',
 'Air temperature (12h mean)',
 'Air temperature (24h max)',
 'Air temperature (24h min)',
 'Air temperature (24h me

In [7]:
# Keep only variables relevant to delay analysis and initial feature investigation
selected_columns = [ 'departureDate',
 'trainType',
 'trainCategory',
 'stationName',
 'type',
 'scheduledTime',
 'differenceInMinutes',
 'Air temperature',
 'Wind speed',
 'Precipitation amount',
  'Snow depth',
  'Horizontal visibility',]

transport_delay_df = data_2025[selected_columns].copy()

In [8]:
transport_delay_df.describe()

,differenceInMinutes,Air temperature,Wind speed,Precipitation amount,Snow depth,Horizontal visibility
count,6.267573e+06,6.284074e+06,6.284060e+06,857521.000000,6.056691e+06,6.283211e+06
mean,3.077412e+00,7.192856e+00,3.738905e+00,0.073598,2.834007e+00,3.766316e+04
std,1.530685e+01,8.884040e+00,2.035406e+00,0.411715,7.781381e+00,2.097869e+04
min,-6.730000e+02,-3.690000e+01,0.000000e+00,0.000000,-1.000000e+00,1.400000e+01
25%,0.000000e+00,9.000000e-01,2.300000e+00,0.000000,-1.000000e+00,2.000000e+04
50%,1.000000e+00,6.500000e+00,3.400000e+00,0.000000,0.000000e+00,4.170500e+04
75%,3.000000e+00,1.380000e+01,4.900000e+00,0.000000,0.000000e+00,5.000000e+04
max,9.300000e+02,3.230000e+01,2.660000e+01,31.600000,7.200000e+01,7.500000e+04


In [9]:
transport_delay_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6284074 entries, 0 to 6284073
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   departureDate          object 
 1   trainType              object 
 2   trainCategory          object 
 3   stationName            object 
 4   type                   object 
 5   scheduledTime          object 
 6   differenceInMinutes    float64
 7   Air temperature        float64
 8   Wind speed             float64
 9   Precipitation amount   float64
 10  Snow depth             float64
 11  Horizontal visibility  float64
dtypes: float64(6), object(6)
memory usage: 575.3+ MB


In [10]:
# Check missing values before deciding how each feature should be handled
transport_delay_df.isna().sum()

departureDate                  0
trainType                      0
trainCategory                  0
stationName                    0
type                           0
scheduledTime                  0
differenceInMinutes        16501
Air temperature                0
Wind speed                    14
Precipitation amount     5426553
Snow depth                227383
Horizontal visibility        863
dtype: int64

## Feature Engineering: Scheduled Times

The scheduled time is originally stored as a full timestamp. To make it usable for machine learning models, we convert it into a single numerical feature: minutes after midnight.

Since `type` (DEPARTURE/ARRIVAL) already tells us which kind of event each row represents, we convert `scheduledTime` directly rather than splitting it into separate departure/arrival columns first — that avoids creating two columns that are each ~50% empty by construction.

In [11]:
transport_delay_df["scheduledTime"] = pd.to_datetime(transport_delay_df["scheduledTime"])

In [12]:
# `type` already tells us whether each row is a DEPARTURE or ARRIVAL event, so there's no need to split scheduledTime into two separate columns and recombine them later.
# scheduledTime is timezone-aware UTC in the source data;convert the recorded timestamp to minutes after midnight.
transport_delay_df["scheduled_time_minutes"] = (
    transport_delay_df["scheduledTime"].dt.hour * 60
    + transport_delay_df["scheduledTime"].dt.minute
)

In [13]:
transport_delay_df[[
    "type",
    "scheduledTime",
    "scheduled_time_minutes"
]].head(20)

,type,scheduledTime,scheduled_time_minutes
0,DEPARTURE,2025-01-01 04:54:00+00:00,294
1,ARRIVAL,2025-01-01 04:59:00+00:00,299
2,DEPARTURE,2025-01-01 05:00:00+00:00,300
3,ARRIVAL,2025-01-01 05:02:00+00:00,302
4,DEPARTURE,2025-01-01 05:02:00+00:00,302
5,ARRIVAL,2025-01-01 05:03:00+00:00,303
6,DEPARTURE,2025-01-01 05:03:00+00:00,303
7,ARRIVAL,2025-01-01 05:04:11+00:00,304
8,DEPARTURE,2025-01-01 05:04:11+00:00,304
9,ARRIVAL,2025-01-01 05:05:00+00:00,305


In [14]:
transport_delay_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6284074 entries, 0 to 6284073
Data columns (total 13 columns):
 #   Column                  Dtype              
---  ------                  -----              
 0   departureDate           object             
 1   trainType               object             
 2   trainCategory           object             
 3   stationName             object             
 4   type                    object             
 5   scheduledTime           datetime64[ns, UTC]
 6   differenceInMinutes     float64            
 7   Air temperature         float64            
 8   Wind speed              float64            
 9   Precipitation amount    float64            
 10  Snow depth              float64            
 11  Horizontal visibility   float64            
 12  scheduled_time_minutes  int64              
dtypes: datetime64[ns, UTC](1), float64(6), int64(1), object(5)
memory usage: 623.3+ MB


## Removing Rows with Missing Delay Values

In [15]:
# Remove rows where the delay information is missing
transport_delay_df = transport_delay_df.dropna(
    subset=["differenceInMinutes"]
).copy()

In [16]:
transport_delay_df["differenceInMinutes"].isna().sum()

np.int64(0)

Rows without a recorded delay cannot be assigned a target value, so they are removed before creating the `delayed` target.

## Creating a Delayed column

In [17]:
# Define a delay as any positive difference in minutes; 0 = on time or early, 1 = delayed
transport_delay_df["delayed"] = (
    transport_delay_df["differenceInMinutes"] > 0
).astype(int)

In [18]:
transport_delay_df["delayed"].value_counts()

delayed
1    4017528
0    2250045
Name: count, dtype: int64

In [19]:
transport_delay_df["delayed"].value_counts(normalize=True) * 100

delayed
1    64.100219
0    35.899781
Name: proportion, dtype: float64

### Finding: Definition of a Delay

The target variable `delayed` is derived from `differenceInMinutes`.

- `delayed = 0` → no positive delay (on time or early)
- `delayed = 1` → positive delay

## Checking on missing Values

In [20]:
transport_delay_df.isna().sum().sort_values(ascending=False)

Precipitation amount      5412756
Snow depth                 226698
Horizontal visibility         829
Wind speed                     14
departureDate                   0
trainType                       0
trainCategory                   0
stationName                     0
type                            0
scheduledTime                   0
differenceInMinutes             0
Air temperature                 0
scheduled_time_minutes          0
delayed                         0
dtype: int64

In [21]:
# Remove the small number of rows missing wind speed or visibility because these variables are retained as model features
transport_delay_df = transport_delay_df.dropna(
    subset=["Wind speed", "Horizontal visibility"]
).copy()

In [22]:
transport_delay_df.isnull().sum()

departureDate                   0
trainType                       0
trainCategory                   0
stationName                     0
type                            0
scheduledTime                   0
differenceInMinutes             0
Air temperature                 0
Wind speed                      0
Precipitation amount      5412007
Snow depth                 226272
Horizontal visibility           0
scheduled_time_minutes          0
delayed                         0
dtype: int64

## Explore missing data for Precipitation

In [23]:
# does missingness cluster by weather station? (use stationName as proxy)
precip_missing_rate = transport_delay_df.groupby("stationName")["Precipitation amount"].apply(lambda x: x.isna().mean())
precip_missing_rate.sort_values(ascending=False).head(20)

stationName
Äänekoski               1.0
Lamminniemi             1.0
Vaskiluoto              1.0
Kyminlinna              1.0
Kymi                    1.0
Valkeakoski             1.0
Nummela                 1.0
Riihimäki lajittelu     1.0
Kouvola tavara          1.0
Kouvola Oikoraide       1.0
Kotkan satama           1.0
Tavastila               1.0
Pieksämäki lajittelu    1.0
Pieksämäki tavara       1.0
Kiukainen               1.0
Pitkäkallio             1.0
Skogby                  1.0
Juurikorpi              1.0
Pännäinen (R855)        1.0
Keitelepohja            1.0
Name: Precipitation amount, dtype: float64

In [24]:
precip_missing_rate.describe()

count    469.000000
mean       0.842219
std        0.165517
min        0.147235
25%        0.751203
50%        0.900000
75%        0.988688
max        1.000000
Name: Precipitation amount, dtype: float64

In [25]:
# check: for a station with LOW missingness, do the non-null values include a lot of zeros, or are they mostly small positive numbers?
low_missing_station = precip_missing_rate.idxmin()  # the 14.7% one
data_2025[data_2025["stationName"] == low_missing_station]["Precipitation amount"].describe()

count    9138.000000
mean        0.074699
std         0.464084
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        15.300000
Name: Precipitation amount, dtype: float64

In [26]:
data_2025["departureDate"] = pd.to_datetime(data_2025["departureDate"])
data_2025["month"] = data_2025["departureDate"].dt.month

In [27]:
# Check whether precipitation missingness follows a seasonal pattern

data_2025.groupby("month")["Precipitation amount"].apply(lambda x: x.isna().mean())

month
1     0.861860
2     0.860977
3     0.860303
4     0.861544
5     0.861411
6     0.858932
7     0.874641
8     0.865869
9     0.864021
10    0.862863
11    0.862704
12    0.867213
Name: Precipitation amount, dtype: float64

In [28]:
# Drop the column completely because it is 86% missing
transport_delay_df = transport_delay_df.drop(columns=["Precipitation amount"])

In [29]:
transport_delay_df.isnull().sum()

departureDate                  0
trainType                      0
trainCategory                  0
stationName                    0
type                           0
scheduledTime                  0
differenceInMinutes            0
Air temperature                0
Wind speed                     0
Snow depth                226272
Horizontal visibility          0
scheduled_time_minutes         0
delayed                        0
dtype: int64

## Conclusions of Missing values in "Precipitation"

The missingness is high across all months, with no clear seasonal pattern. Combined with the presence of recorded zero values, this suggests that missing values should not automatically be interpreted as zero precipitation.

We therefore do not replace missing values with 0, as this would treat unavailable measurements as confirmed no-precipitation observations.

Given that approximately 86% of precipitation observations are missing, the column is dropped rather than imputed.

## Analysis of Missing Snow Depth Values

The `Snow depth` column contains both missing values (`NaN`) and a value of `-1`.

First, we examine the distribution of missing values across the year to determine whether the missingness follows a seasonal pattern.

We then investigate the `-1` values by comparing their monthly distribution with observations where `Snow depth` is recorded as `0`.

The purpose of this analysis is to distinguish between:

- `NaN` → missing or unavailable measurement
- `0` → recorded zero snow depth
- `-1` → a potential sentinel value requiring further investigation

Based on the observed patterns, we will decide how to handle these values before using `Snow depth` as a feature in the model.

In [30]:
transport_delay_df["Snow depth"].isna().sum()

np.int64(226272)

In [31]:
transport_delay_df["Snow depth"].describe()

count    6.040458e+06
mean     2.834911e+00
std      7.781873e+00
min     -1.000000e+00
25%     -1.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      7.200000e+01
Name: Snow depth, dtype: float64

In [32]:
# Check the values in Snow depth
transport_delay_df["Snow depth"].value_counts().sort_index().head(10)

Snow depth
-1.0    2866011
 0.0    1721323
 2.0     130034
 3.0      92174
 4.0      73672
 5.0      52055
 6.0      46170
 7.0      46828
 8.0      63040
 9.0      68781
Name: count, dtype: int64

In [33]:
transport_delay_df["departureDate"] = pd.to_datetime(
    transport_delay_df["departureDate"]
)

In [34]:
(transport_delay_df["Snow depth"] == -1).groupby(
    transport_delay_df["departureDate"].dt.month
).mean()

departureDate
1     0.000000
2     0.000000
3     0.000000
4     0.000011
5     0.450745
6     0.966659
7     0.964844
8     0.968270
9     0.968468
10    0.917441
11    0.320741
12    0.000044
Name: Snow depth, dtype: float64

In [35]:
(transport_delay_df["Snow depth"] == 0).groupby(
    transport_delay_df["departureDate"].dt.month
).mean()

departureDate
1     0.046197
2     0.124316
3     0.467838
4     0.900591
5     0.508014
6     0.000442
7     0.000000
8     0.000000
9     0.000031
10    0.050945
11    0.450445
12    0.706510
Name: Snow depth, dtype: float64

In [36]:
transport_delay_df.groupby(
    transport_delay_df["departureDate"].dt.month
)["Snow depth"].apply(
    lambda x: x.isna().mean()
)

departureDate
1     0.041815
2     0.040931
3     0.041704
4     0.041690
5     0.039716
6     0.032899
7     0.035156
8     0.031730
9     0.031501
10    0.031562
11    0.032881
12    0.031634
Name: Snow depth, dtype: float64

## Analysis of Snow Depth Sentinel Value (-1)

`Snow depth` contains a large number of `-1` values (~2.87M rows, ~45% of the dataset)
alongside genuine `0` and positive readings. Snow depth cannot physically be negative,
so `-1` is not a real measurement and needs to be understood before it can be used in
analysis or modelling.

Checking the monthly distribution of `-1` versus `0` reveals a clear seasonal pattern:

- `-1` is effectively absent from January–April and December (0.00–0.05%), then rises
  sharply to near-100% across June–September, with May, October, and November as
  transition months.
- `0` shows almost exactly the opposite pattern: dominant in the winter/spring months
  when `-1` is absent, and close to 0% in the months when `-1` dominates.

In every month, the two values are nearly mutually exclusive. This strong seasonal
relationship suggests that suggests that `-1` is being used as a sentinel value associated with periods of no recorded snow depth

**Conclusion:** Based on the observed seasonal pattern, we treat `-1` as equivalent to 0 for this analysis.

In [37]:
# Convert the -1 sentinel value to 0 based on the observed seasonal pattern
transport_delay_df["Snow depth"] = transport_delay_df["Snow depth"].replace(-1, 0)

In [38]:
transport_delay_df["Snow depth"].value_counts().sort_index().head(10)

Snow depth
0.0     4587334
2.0      130034
3.0       92174
4.0       73672
5.0       52055
6.0       46170
7.0       46828
8.0       63040
9.0       68781
10.0      54535
Name: count, dtype: int64

In [39]:
transport_delay_df["Snow depth"].isna().sum()


np.int64(226272)

In [40]:
transport_delay_df["Snow depth"].isna().mean()

np.float64(0.036106869132705575)

In [41]:
# Remove the remaining rows with unavailable snow-depth measurements
transport_delay_df = transport_delay_df.dropna(subset=["Snow depth"]).copy()

In [42]:
transport_delay_df["Snow depth"].isna().sum()

np.int64(0)

## Fix: keep `type` as a feature

Without `type`, the model has no way to know whether `scheduled_time_minutes` refers to an arrival or a departure — and arrival delay (accumulated along the route) behaves differently from departure delay (resets at each station). Encode it as a binary flag rather than dropping it.

In [43]:
print(transport_delay_df["type"].unique())

transport_delay_df["type_encoded"] = transport_delay_df["type"].map({"DEPARTURE": 0, "ARRIVAL": 1})
transport_delay_df["type_encoded"].isna().sum()  # should be 0 - confirms the map covered every value

['DEPARTURE' 'ARRIVAL']


np.int64(0)

## Fix: turn `departureDate` into usable numeric features

A raw datetime column can't be fed into scikit-learn models directly. Extract calendar features that can plausibly affect delay (month for seasonal weather effects, day of week for traffic/schedule density) instead of the timestamp itself.

In [44]:
transport_delay_df["departureDate"] = pd.to_datetime(transport_delay_df["departureDate"])
transport_delay_df["month"] = transport_delay_df["departureDate"].dt.month
transport_delay_df["day_of_week"] = transport_delay_df["departureDate"].dt.dayofweek  # 0=Monday

## Station Name Cardinality Check

`stationName` has 466 unique values. This relatively high cardinality makes direct one-hot encoding less attractive because it would substantially increase the feature space.

We therefore assess whether station-level information is within the scope of this analysis before deciding how to handle the column.

In [45]:
transport_delay_df["stationName"].nunique()

466

In [46]:
transport_delay_df["stationName"].value_counts().describe()

count      466.000000
mean     12962.356223
std      15625.383742
min          1.000000
25%       1540.000000
50%       8291.500000
75%      18068.000000
max      83582.000000
Name: count, dtype: float64

## Decision: Dropping `stationName`

`stationName` has 466 unique values, and while station is likely a meaningful predictor of delay
in its own right (junction complexity, platform congestion, etc.), it falls outside the scope of
this analysis. The research question here is specifically about the relationship between weather
and delay, not station-level operational effects — so `stationName` is excluded rather than encoded.

This is a scope decision, not a performance-driven one: no comparison was run to test whether
including station (via target or frequency encoding) would improve predictive performance. If a
later goal is prediction accuracy rather than weather-effect analysis, this column would be worth
revisiting.

In [47]:
transport_delay_df = transport_delay_df.drop(
    columns=["stationName"]
)

In [48]:
"stationName" in transport_delay_df.columns

False

## Selecting Features and Target

The target is `delayed`.

The modelling feature set uses the engineered numeric variables:
- calendar features
- event type
- scheduled time
- cyclical time features
- weather variables

`stationName` is excluded because station-level operational effects are outside the scope of this analysis.

In [49]:
# Encode time-related variables cyclically so adjacent values remain close
# (e.g. 23:59 is close to 00:00)

# Make sure the input columns are numeric
transport_delay_df["scheduled_time_minutes"] = pd.to_numeric(
    transport_delay_df["scheduled_time_minutes"],
    errors="coerce"
)

transport_delay_df["month"] = pd.to_numeric(
    transport_delay_df["month"],
    errors="coerce"
)

transport_delay_df["day_of_week"] = pd.to_numeric(
    transport_delay_df["day_of_week"],
    errors="coerce"
)

# Cyclical encoding for time of day
transport_delay_df["sin_time"] = np.sin(
    2 * np.pi * transport_delay_df["scheduled_time_minutes"] / 1440
)

transport_delay_df["cos_time"] = np.cos(
    2 * np.pi * transport_delay_df["scheduled_time_minutes"] / 1440
)

# Cyclical encoding for month
transport_delay_df["sin_month"] = np.sin(
    2 * np.pi * (transport_delay_df["month"] - 1) / 12
)

transport_delay_df["cos_month"] = np.cos(
    2 * np.pi * (transport_delay_df["month"] - 1) / 12
)

# Cyclical encoding for day of week
transport_delay_df["sin_day"] = np.sin(
    2 * np.pi * transport_delay_df["day_of_week"] / 7
)

transport_delay_df["cos_day"] = np.cos(
    2 * np.pi * transport_delay_df["day_of_week"] / 7
)


## Undersampling to balance the data

In [50]:
# Separate features and target
# Select features
X = transport_delay_df[[
    "month",
    "day_of_week",
    "type_encoded",
    "scheduled_time_minutes",
    "Air temperature",
    "Wind speed",
    "Snow depth",
    "Horizontal visibility",
    "sin_time",
    "cos_time",
    "sin_month",
    "cos_month",
    "sin_day",
    "cos_day"
]]

y = transport_delay_df["delayed"]

# Split into training and test sets BEFORE undersampling
# Keep the test set untouched so final performance reflects the original class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

# Undersample TRAINING data only
# Balance the training set to reduce majority-class dominance
rus = RandomUnderSampler(random_state=SEED)

X_train_resampled, y_train_resampled = rus.fit_resample(
    X_train,
    y_train
)

print("\nTraining data after undersampling:")
print(X_train_resampled.shape)
print(y_train_resampled.value_counts())
print("\nTraining class balance:")
print(y_train_resampled.value_counts(normalize=True) * 100)

X_train: (4832366, 14)
X_test: (1208092, 14)
y_train: (4832366,)
y_test: (1208092,)

Training data after undersampling:
(3484132, 14)
delayed
0    1742066
1    1742066
Name: count, dtype: int64

Training class balance:
delayed
0    50.0
1    50.0
Name: proportion, dtype: float64


In [51]:
print("Test set class distribution:")
print(y_test.value_counts())

print("\nTest set class proportions:")
print(y_test.value_counts(normalize=True) * 100)

Test set class distribution:
delayed
1    772575
0    435517
Name: count, dtype: int64

Test set class proportions:
delayed
1    63.950014
0    36.049986
Name: proportion, dtype: float64


## Exporting the dataframe for ML

In [52]:
# Export the balanced training set and untouched test set for model development and evaluation

# Combine undersampled training features and target
df_train_resampled = pd.DataFrame(
    X_train_resampled,
    columns=X.columns
)

df_train_resampled["delayed"] = y_train_resampled

# Combine untouched test features and target
df_test = pd.DataFrame(
    X_test,
    columns=X.columns
)

df_test["delayed"] = y_test

# Export training and test datasets
df_train_resampled.to_csv(
    "train_undersampled.csv",
    index=False
)

df_test.to_csv(
    "test.csv",
    index=False
)

print("Training dataset exported:", df_train_resampled.shape)
print("Test dataset exported:", df_test.shape)

Training dataset exported: (3484132, 15)
Test dataset exported: (1208092, 15)
